In [36]:
# %pip install msal
import os
import requests
from msal import ConfidentialClientApplication
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 11
python-dotenv could not parse statement starting at line 14


True

In [37]:
TENANT_ID     = os.environ["CRM_TENANT_ID"]
CLIENT_ID     = os.environ["CRM_CLIENT_ID"]
CLIENT_SECRET = os.environ["CRM_CLIENT_SECRET"]
ENVIRONMENT_URL = os.environ["CRM_ENVIRONMENT_URL"]

In [58]:
FETCHXML = """
<fetch version="1.0" output-format="xml-platform" mapping="logical" savedqueryid="0d5d377b-5e7c-47b5-bab1-a5cb8b4ac105" no-lock="false" distinct="true">
  <entity name="contact">
    <attribute name="statecode"/>
    <attribute name="entityimage_url"/>
    <attribute name="fullname"/>
    <attribute name="emailaddress1"/>
    <attribute name="telephone1"/>
    <attribute name="contactid"/>
    <attribute name="vgr_contacttypes"/>
    <attribute name="mobilephone"/>
    <filter type="and">
      <condition attribute="statecode" operator="eq" value="0"/>
    </filter>
    <link-entity name="account" from="accountid" to="parentcustomerid" link-type="inner" alias="a_ce3f647c660e4a5c99cc3d631f23406d">
        <attribute name="accountnumber"/>
      <filter type="and">
        <condition attribute="accountcategorycode" operator="eq" value="1"/>
      </filter>
    </link-entity>
      <attribute name="firstname"/>
      <attribute name="lastname"/>
  </entity>
</fetch>
"""

In [59]:
def get_access_token() -> str:
    app = ConfidentialClientApplication(
        client_id=CLIENT_ID,
        client_credential=CLIENT_SECRET,
        authority=f"https://login.microsoftonline.com/{TENANT_ID}",
    )
    result = app.acquire_token_for_client(scopes=[f"{ENVIRONMENT_URL}/.default"])
    if "access_token" not in result:
        raise RuntimeError(f"Token acquisition failed: {result.get('error_description')}")
    return result["access_token"]

In [66]:
def get_contacts(token: str) -> pd.DataFrame:
    """Fetch Consumer contacts from Dataverse using FetchXML and return as a DataFrame."""
    headers = {
        "Authorization": f"Bearer {token}",
        "OData-MaxVersion": "4.0",
        "OData-Version": "4.0",
        "Accept": "application/json",
        "Prefer": 'odata.maxpagesize=5000, odata.include-annotations="Microsoft.Dynamics.CRM.*"',
    }

    all_records = []
    page = 1
    paging_cookie = None

    while True:
        # Inject paging cookie into FetchXML after the first page
        if paging_cookie:
            fetch = FETCHXML.replace(
                '<fetch version="1.0"',
                f'<fetch version="1.0" page="{page}" paging-cookie="{paging_cookie}"'
            )
        else:
            fetch = FETCHXML

        url = f"{ENVIRONMENT_URL}/api/data/v9.2/contacts?fetchXml={urllib.parse.quote(fetch)}"
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()

        print("More records:", data.get("@Microsoft.Dynamics.CRM.morerecords"))
        print("Paging cookie:", data.get("@Microsoft.Dynamics.CRM.pagingcookie"))
        print("Records this page:", len(data.get("value", [])))
        print("All response keys:", list(data.keys()))

        records = data.get("value", [])
        all_records.extend(records)
        print(f"Page {page}: fetched {len(records)} records (total so far: {len(all_records):,})")

        # Check if there are more pages
        if data.get("@Microsoft.Dynamics.CRM.morerecords"):
            # FetchXML pagination uses fetchxmlpagingcookie, not pagingcookie
            raw_cookie = data.get("@Microsoft.Dynamics.CRM.fetchxmlpagingcookie") or \
                        data.get("@Microsoft.Dynamics.CRM.pagingcookie")
    
        if not raw_cookie:
            print("Warning: morerecords is True but no paging cookie returned — stopping pagination")
            break
                
            paging_cookie = urllib.parse.quote(raw_cookie)
            page += 1
        else:
            break

    df = pd.DataFrame(all_records)

    # Rename aliased account fields (aa.* prefix from the link-entity alias)
    df.rename(columns=lambda c: c.replace("aa.", "account_"), inplace=True)

    print(f"\nDone — {len(df):,} Consumer contacts loaded into DataFrame")
    print(f"Columns: {df.columns.tolist()}")
    return df


if __name__ == "__main__":
    token = get_access_token()
    df = get_contacts(token)
    print(df.head(500))

More records: True
Paging cookie: None
Records this page: 5000
All response keys: ['@odata.context', '@Microsoft.Dynamics.CRM.totalrecordcount', '@Microsoft.Dynamics.CRM.totalrecordcountlimitexceeded', '@Microsoft.Dynamics.CRM.globalmetadataversion', '@Microsoft.Dynamics.CRM.fetchxmlpagingcookie', '@Microsoft.Dynamics.CRM.morerecords', 'value']
Page 1: fetched 5000 records (total so far: 5,000)

Done — 5,000 Consumer contacts loaded into DataFrame
Columns: ['@odata.etag', 'mobilephone', 'contactid', 'statecode', 'lastname', 'firstname', 'fullname', 'emailaddress1', 'vgr_contacttypes', 'a_ce3f647c660e4a5c99cc3d631f23406d.accountnumber', 'telephone1']
        @odata.etag   mobilephone                             contactid  \
0    W/"3310522982"  021-225-5031  d148896d-08d5-ee11-904c-000d3a6a016e   
1    W/"5331737793"           NaN  c2429943-abd5-ee11-904c-000d3a6a016e   
2    W/"5331749717"           NaN  d9a91307-87dd-ee11-904c-000d3a6a016e   
3    W/"5332759973"           NaN  610d1a9

In [41]:
# Mapping from Dataverse values to OneBill contact type labels
CONTACT_TYPE_MAP = {
    "287790000": "Billing",
    "287790001": "Technical",
    "287790002": "Outage - Email",
    "287790009": "Outage - SMS",
    "287790003": "Primary",
    "287790004": "Technical - Data",
    "287790005": "Technical - Voice",
    "287790006": "Commercial",
    "287790008": "Communication",
    "287790007": "Voyager Staff",
}

# Contact types that are kept together as communication preferences, not split into OneBill rows
COMMUNICATION_PREFS = {"Billing", "Primary"}

In [53]:
def expand_contacts(df: pd.DataFrame) -> pd.DataFrame:
    """
    Splits vgr_contacttypes into individual rows:
    - Billing and Primary are combined into one row as communication_preferences
      with is_billing and is_primary flags set to True
    - All other contact types get their own row with is_billing/is_primary
      set to reflect whether the original contact had those preferences
    """
    expanded_rows = []

    for _, row in df.iterrows():
        raw = str(row.get("vgr_contacttypes", "") or "")

        # Parse semicolon-separated values and map to labels
        type_values = [v.strip() for v in raw.split(",") if v.strip()]
        type_labels = [CONTACT_TYPE_MAP.get(v, v) for v in type_values]

        is_billing = "Billing" in type_labels
        is_primary = "Primary" in type_labels

        onebill_types = [t for t in type_labels if t not in COMMUNICATION_PREFS]

        base = {
            **row.to_dict(),
            "is_billing": is_billing,
            "is_primary": is_primary,
        }

        # Row 1 — communication preferences row (only if Billing or Primary is present)
        if is_billing or is_primary:
            comm_prefs = ", ".join(t for t in ["Billing", "Primary"] if t in type_labels)
            expanded_rows.append({
                **base,
                "contact_type": comm_prefs,
            })

        # Additional rows — one per OneBill contact type, is_billing/is_primary set to False
        for contact_type in onebill_types:
            expanded_rows.append({
                **base,
                "contact_type": contact_type,
                "is_billing": False,
                "is_primary": False,
            })

        # If no types at all, keep the contact with a single empty row
        if not is_billing and not is_primary and not onebill_types:
            expanded_rows.append({
                **base,
                "contact_type": None,
            })

    result = pd.DataFrame(expanded_rows)
    result.drop(columns=["vgr_contacttypes"], inplace=True, errors="ignore")
    result.reset_index(drop=True, inplace=True)

    return result

In [54]:
df_expanded = expand_contacts(df)

print(f"Original rows:  {len(df):,}")
print(f"Expanded rows:  {len(df_expanded):,}")
print(df_expanded[["fullname", "contact_type", "is_billing", "is_primary"]].head(20))

Original rows:  5,000
Expanded rows:  5,000
                 fullname contact_type  is_billing  is_primary
0              David Earl          nan       False       False
1            Garry Martin      Billing        True       False
2           Justin Bagust      Billing        True       False
3            Shanaya Keil      Billing        True       False
4               James Suh          nan       False       False
5          Michelle Child      Billing        True       False
6        Te Ara Bergstrom          nan       False       False
7   Jonathan Clark-Howard      Billing        True       False
8           Hunter McLeod      Billing        True       False
9            Lennart Nout      Billing        True       False
10               Alf Mohi      Billing        True       False
11            Gene Hanham      Billing        True       False
12             Judy Sibbe          nan       False       False
13         Tracker Apiata      Billing        True       False
14         